In [ ]:
# membandit setup
import os
import sys
from pathlib import Path

# Ensure cleaned_repo (parent of membandit/) is on sys.path when running from notebooks/
_here = Path.cwd().resolve()
for _candidate in [_here, *_here.parents]:
    if (_candidate / "membandit" / "__init__.py").exists():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

# Optional: set MEMORYBANDIT_DATA_DIR to your data root (defaults to /Volumes/T7 Shield)
# os.environ.setdefault("MEMORYBANDIT_DATA_DIR", "/path/to/data/root")

from membandit.plotting_style import set_plot_style
set_plot_style()


# Cue-Trial HPC->ACC Theta GC Predicting ACC Reinstatement

Exploratory notebook with inline smoke-subject GC internals and a resumable batch expansion.


In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

import os
import json
import warnings
from copy import deepcopy

import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from mne.stats import permutation_cluster_1samp_test
from scipy.interpolate import RegularGridInterpolator

from membandit.plotting_style import set_plot_style
from membandit.behavior import load_behavior_tables, preprocess_learn_df, preprocess_combined_df
from membandit.io import load_subject_epochs
from membandit._impl.multivariate import TFRClusterPermutationGLM

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(x, **kwargs):
        return x

try:
    from mne_connectivity import spectral_connectivity_time
except Exception:
    spectral_connectivity_time = None

warnings.filterwarnings('ignore')
set_plot_style()
np.set_printoptions(suppress=True, precision=4)


In [ ]:
# Paths, subjects, behavior
base_dir = '/Volumes/T7 Shield'
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion == 1) & (demo_data.Minerva == 1)].copy()
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]
subjects = sorted(subj_df['MSSMCode'].dropna().astype(str).unique().tolist())

learn_df_raw, combined_df_raw = load_behavior_tables(load_dir)
learn_df = preprocess_learn_df(learn_df_raw)
combined_df = preprocess_combined_df(combined_df_raw)

print(f'Found {len(subjects)} analyzable subjects')
print('learn_df shape:', learn_df.shape)
print('combined_df shape:', combined_df.shape)


In [ ]:
# Config
GC_CFG = {
    'event': 'cue_on',
    'day': 'Day1',
    'roi_source': 'HPC',
    'roi_target': 'ACC',
    'tmin': 0.0,
    'tmax': 1.0,
    'target_sfreq': 50,
    'fmin': 4.0,
    'fmax': 8.0,
    'n_cycles': 2,
    'gc_n_lags': 10,
    'n_surrogates_smoke': 100,
    'n_surrogates_batch': 100,
    'random_state': 42,
}

MV_CFG = {
    'roi': 'ACC',
    'predictor': 'rpe_cat',
    'trial_save_dirs': [
        '/Volumes/T7 Shield/scratch/MemoryBandit/RPE_Reactivation/trial_level/multivariate',
        '/Volumes/T7 Shield/scratch/MemoryBandit/RPE_Reactivation/trial_level',
    ],
}

GLM_CFG = {
    'predictor_col': 'gc_net_z',
    'alternative': 'two-sided',
    'alpha_cluster': 0.05,
    'alpha_test': 0.05,
    'min_trials': 15,
    'n_permutations_smoke': 500,
    'n_permutations_batch': 2000,
    'n_permutations_group': 5000,
    'seed': 42,
}

RUN = {
    'smoke_subject': 'MS036' if 'MS036' in subjects else subjects[0],
}

analysis_tag = f"{GC_CFG['event']}_{GC_CFG['roi_source']}to{GC_CFG['roi_target']}_theta_{int(GC_CFG['fmin'])}-{int(GC_CFG['fmax'])}_{MV_CFG['roi']}_{MV_CFG['predictor']}"
analysis_dir = os.path.join(base_dir, 'projects', 'guLab', 'Salman', 'EphysAnalyses', 'Connectivity', 'scratch', 'cue_trial_gc_acc_reinstatement', analysis_tag)
batch_cache_dir = os.path.join(analysis_dir, 'batch_gc_cache')
tables_dir = os.path.join(analysis_dir, 'tables')
plots_dir = os.path.join(analysis_dir, 'plots')
for path in [analysis_dir, batch_cache_dir, tables_dir, plots_dir]:
    os.makedirs(path, exist_ok=True)

gc_trial_df = pd.DataFrame()
mv_trial_df = pd.DataFrame()
merged_trial_df = pd.DataFrame()
subject_glm_results = {}
group_glm_summary = {}

print('Smoke subject:', RUN['smoke_subject'])
print('Analysis dir:', analysis_dir)


In [ ]:
# Lightweight helpers only
def resolve_mv_npz(subject, roi, predictor, trial_save_dirs):
    candidates = []
    for root in trial_save_dirs:
        candidates.extend([
            (os.path.join(root, roi, f'{subject}_{predictor}_zreactivation.npz'), 'new_style' if root.endswith('/multivariate') else 'legacy_predictor'),
            (os.path.join(root, roi, f'{subject}_zreactivation.npz'), 'legacy_unnamed'),
        ])
    seen = set()
    for path, convention in candidates:
        if path in seen:
            continue
        seen.add(path)
        if os.path.exists(path):
            return path, convention
    return None, None

def extract_trial_gc_vector(con_obj, n_epochs, label):
    raw = np.asarray(con_obj.get_data())
    # print(f'{label} raw shape:', raw.shape)
    arr = np.squeeze(raw)
    if arr.ndim == 1 and arr.shape[0] == n_epochs:
        return arr.astype(float)
    if arr.ndim == 2 and arr.shape[0] == n_epochs and arr.shape[1] == 1:
        return arr[:, 0].astype(float)
    if arr.ndim == 2 and arr.shape[1] == n_epochs and arr.shape[0] == 1:
        return arr[0].astype(float)
    raise RuntimeError(f'{label}: expected one value per trial, got shape {arr.shape}')

def scalar_null_stats(real_value, surrogate_values, eps=1e-8):
    surrogate_values = np.asarray(surrogate_values, dtype=float)
    surrogate_values = surrogate_values[np.isfinite(surrogate_values)]
    if surrogate_values.size == 0 or not np.isfinite(real_value):
        return dict(mean=np.nan, std=np.nan, z=np.nan, p=np.nan, n=0)
    mean_val = float(np.nanmean(surrogate_values))
    std_val = float(max(np.nanstd(surrogate_values), eps))
    p_val = (np.sum(surrogate_values >= float(real_value)) + 1.0) / (surrogate_values.size + 1.0)
    return dict(mean=mean_val, std=std_val, z=float((real_value - mean_val) / std_val), p=float(p_val), n=int(surrogate_values.size))

def contiguous_runs(mask):
    runs = []
    i = 0
    while i < len(mask):
        if mask[i]:
            j = i
            while j < len(mask) and mask[j]:
                j += 1
            runs.append((i, j - 1))
            i = j
        else:
            i += 1
    return runs

def compute_subject_gc_cache(subject, n_surrogates, verbose=False):
    epoch_data = load_subject_epochs(
        subj_id=subject,
        base_dir=base_dir,
        event=GC_CFG['event'],
        roi_source=GC_CFG['roi_source'],
        roi_target=GC_CFG['roi_target'],
        tmin=GC_CFG['tmin'],
        tmax=GC_CFG['tmax'],
        target_sfreq=GC_CFG['target_sfreq'],
        day=GC_CFG['day'],
        fmin=GC_CFG['fmin'],
        fmax=GC_CFG['fmax'],
    )
    if epoch_data is None:
        return pd.DataFrame(), {'subject': subject, 'skip_reason': 'epoch_load_failed'}

    epochs = epoch_data['epochs']
    freqs = np.linspace(GC_CFG['fmin'], GC_CFG['fmax'], 8)
    trial_col = 'trials_mem' if 'trials_mem' in combined_df.columns else ('trials_gamble' if 'trials_gamble' in combined_df.columns else 'trials')
    trial_map = combined_df[(combined_df['participant'] == subject) & (combined_df['condition'] == GC_CFG['day'])].copy()
    trial_map = trial_map.dropna(subset=[trial_col]).copy()
    trial_map['trial_index'] = pd.to_numeric(trial_map[trial_col], errors='coerce').astype(int) - 1
    trial_map = trial_map[(trial_map['trial_index'] >= 0) & (trial_map['trial_index'] < len(epochs))].copy()
    trial_map = trial_map.sort_values('trial_index').drop_duplicates(subset=['trial_index'], keep='first').reset_index(drop=True)
    if len(trial_map) == 0:
        return pd.DataFrame(), {'subject': subject, 'skip_reason': 'no_aligned_trials'}

    raw_by_hemi = {}
    indices_fwd_by_hemi = {}
    indices_bwd_by_hemi = {}
    for hemi in ('l', 'r'):
        src_idx = np.unique(np.asarray(epoch_data.get(f'source_indices_{hemi}', []), dtype=int))
        tgt_idx = np.unique(np.asarray(epoch_data.get(f'target_indices_{hemi}', []), dtype=int))
        if src_idx.size == 0 or tgt_idx.size == 0:
            continue
        indices_fwd = (np.array([src_idx.tolist()]), np.array([tgt_idx.tolist()]))
        indices_bwd = (np.array([tgt_idx.tolist()]), np.array([src_idx.tolist()]))
        indices_fwd_by_hemi[hemi] = indices_fwd
        indices_bwd_by_hemi[hemi] = indices_bwd
        gc_fwd = spectral_connectivity_time(
            epochs,
            sfreq=epochs.info['sfreq'],
            method=['gc'],
            indices=indices_fwd,
            fmin=GC_CFG['fmin'],
            fmax=GC_CFG['fmax'],
            freqs=freqs,
            n_cycles=GC_CFG['n_cycles'],
            gc_n_lags=GC_CFG['gc_n_lags'],
            faverage=True,
            verbose='ERROR',
        )
        gc_bwd = spectral_connectivity_time(
            epochs,
            sfreq=epochs.info['sfreq'],
            method=['gc'],
            indices=indices_bwd,
            fmin=GC_CFG['fmin'],
            fmax=GC_CFG['fmax'],
            freqs=freqs,
            n_cycles=GC_CFG['n_cycles'],
            gc_n_lags=GC_CFG['gc_n_lags'],
            faverage=True,
            verbose='ERROR',
        )
        raw_by_hemi[hemi] = {
            'gc_forward_theta_all': extract_trial_gc_vector(gc_fwd, len(epochs), f'{subject} {hemi} forward'),
            'gc_backward_theta_all': extract_trial_gc_vector(gc_bwd, len(epochs), f'{subject} {hemi} backward'),
        }
        raw_by_hemi[hemi]['gc_net_theta_all'] = raw_by_hemi[hemi]['gc_forward_theta_all'] - raw_by_hemi[hemi]['gc_backward_theta_all']

    if len(raw_by_hemi) == 0:
        return pd.DataFrame(), {'subject': subject, 'skip_reason': 'no_valid_hemispheres'}

    base_data = epochs.get_data(copy=True)
    rng = np.random.default_rng(GC_CFG['random_state'])
    min_shift = max(GC_CFG['gc_n_lags'] + 1, 2)
    surrogates_by_hemi = {}
    for hemi in raw_by_hemi:
        src_idx = np.unique(np.asarray(epoch_data[f'source_indices_{hemi}'], dtype=int))
        indices_fwd = indices_fwd_by_hemi[hemi]
        indices_bwd = indices_bwd_by_hemi[hemi]
        n_trials = base_data.shape[0]
        n_times = base_data.shape[-1]
        candidate_shifts = np.arange(min_shift, n_times - min_shift) if (n_times - min_shift) > min_shift else np.arange(1, n_times)
        if candidate_shifts.size == 0:
            candidate_shifts = np.arange(1, n_times)
        surr_forward = np.full((n_surrogates, n_trials), np.nan, dtype=float)
        surr_backward = np.full((n_surrogates, n_trials), np.nan, dtype=float)
        surr_net = np.full((n_surrogates, n_trials), np.nan, dtype=float)
        iterator = range(n_surrogates)
        if verbose:
            iterator = tqdm(iterator, desc=f'{subject} {hemi} surrogates')
        for s_idx in iterator:
            shifted_data = base_data.copy()
            for trial_idx in range(n_trials):
                shift = int(rng.choice(candidate_shifts))
                shifted_data[trial_idx, src_idx, :] = np.roll(shifted_data[trial_idx, src_idx, :], shift=shift, axis=-1)
            shifted_epochs = mne.EpochsArray(
                shifted_data,
                info=deepcopy(epochs.info),
                events=epochs.events.copy(),
                event_id=epochs.event_id.copy() if epochs.event_id is not None else None,
                tmin=float(epochs.tmin),
                baseline=epochs.baseline,
                verbose=False,
            )
            surr_fwd = spectral_connectivity_time(
                shifted_epochs,
                sfreq=shifted_epochs.info['sfreq'],
                method=['gc'],
                indices=indices_fwd,
                fmin=GC_CFG['fmin'],
                fmax=GC_CFG['fmax'],
                freqs=freqs,
                n_cycles=GC_CFG['n_cycles'],
                gc_n_lags=GC_CFG['gc_n_lags'],
                faverage=True,
                verbose='ERROR',
            )
            surr_bwd = spectral_connectivity_time(
                shifted_epochs,
                sfreq=shifted_epochs.info['sfreq'],
                method=['gc'],
                indices=indices_bwd,
                fmin=GC_CFG['fmin'],
                fmax=GC_CFG['fmax'],
                freqs=freqs,
                n_cycles=GC_CFG['n_cycles'],
                gc_n_lags=GC_CFG['gc_n_lags'],
                faverage=True,
                verbose='ERROR',
            )
            surr_fwd_trial = extract_trial_gc_vector(surr_fwd, len(shifted_epochs), f'{subject} {hemi} surr forward {s_idx}')
            surr_bwd_trial = extract_trial_gc_vector(surr_bwd, len(shifted_epochs), f'{subject} {hemi} surr backward {s_idx}')
            surr_forward[s_idx] = surr_fwd_trial
            surr_backward[s_idx] = surr_bwd_trial
            surr_net[s_idx] = surr_fwd_trial - surr_bwd_trial
        surrogates_by_hemi[hemi] = {'surr_forward': surr_forward, 'surr_backward': surr_backward, 'surr_net': surr_net}

    rows = []
    for _, row in trial_map.iterrows():
        trial_idx = int(row['trial_index'])
        hemi_rows = []
        for hemi in raw_by_hemi:
            real_forward = float(raw_by_hemi[hemi]['gc_forward_theta_all'][trial_idx])
            real_backward = float(raw_by_hemi[hemi]['gc_backward_theta_all'][trial_idx])
            real_net = float(raw_by_hemi[hemi]['gc_net_theta_all'][trial_idx])
            stats_fwd = scalar_null_stats(real_forward, surrogates_by_hemi[hemi]['surr_forward'][:, trial_idx])
            stats_bwd = scalar_null_stats(real_backward, surrogates_by_hemi[hemi]['surr_backward'][:, trial_idx])
            stats_net = scalar_null_stats(real_net, surrogates_by_hemi[hemi]['surr_net'][:, trial_idx])
            hemi_rows.append({
                'gc_forward_theta': real_forward,
                'gc_backward_theta': real_backward,
                'gc_net_theta': real_net,
                'gc_forward_z': stats_fwd['z'],
                'gc_backward_z': stats_bwd['z'],
                'gc_net_z': stats_net['z'],
                'p_forward': stats_fwd['p'],
                'p_backward': stats_bwd['p'],
                'p_net': stats_net['p'],
                'n_surrogates': stats_net['n'],
            })
        hemi_df = pd.DataFrame(hemi_rows)
        rows.append({
            'subject': subject,
            'trial_index': trial_idx,
            'trials_mem': int(row[trial_col]),
            'rpe': row['rpe'] if 'rpe' in row else np.nan,
            'rpe_cat': row['rpe_cat'] if 'rpe_cat' in row else np.nan,
            'memory_outcome': row['memory_outcome'] if 'memory_outcome' in row else np.nan,
            'gc_forward_theta': float(hemi_df['gc_forward_theta'].mean()),
            'gc_backward_theta': float(hemi_df['gc_backward_theta'].mean()),
            'gc_net_theta': float(hemi_df['gc_net_theta'].mean()),
            'gc_forward_z': float(hemi_df['gc_forward_z'].mean()),
            'gc_backward_z': float(hemi_df['gc_backward_z'].mean()),
            'gc_net_z': float(hemi_df['gc_net_z'].mean()),
            'p_forward': float(hemi_df['p_forward'].mean()),
            'p_backward': float(hemi_df['p_backward'].mean()),
            'p_net': float(hemi_df['p_net'].mean()),
            'valid_hemi_count': int(len(hemi_df)),
            'n_surrogates': int(np.round(hemi_df['n_surrogates'].mean())),
            'gc_success': True,
        })
    out_df = pd.DataFrame(rows).sort_values('trial_index').reset_index(drop=True)
    out_df['gc_net_theta'] = out_df['gc_forward_theta'] - out_df['gc_backward_theta']
    meta = {
        'subject': subject,
        'skip_reason': '',
        'valid_hemispheres': sorted(list(raw_by_hemi.keys())),
        'n_trials': int(len(out_df)),
    }
    return out_df, meta

def load_subject_mv_payload(subject):
    mv_path, convention = resolve_mv_npz(subject, MV_CFG['roi'], MV_CFG['predictor'], MV_CFG['trial_save_dirs'])
    if mv_path is None:
        return None
    npz = np.load(mv_path, allow_pickle=True)
    meta = {k: npz[k] for k in npz.files if k not in {'z_reactivation', 'projection_real', 'feedback_times', 'cue_times'}}
    z_reactivation = np.asarray(npz['z_reactivation'])
    projection_real = np.asarray(npz['projection_real']) if 'projection_real' in npz.files else None
    feedback_times = np.asarray(npz['feedback_times']) if 'feedback_times' in npz.files else np.arange(z_reactivation.shape[0])
    cue_times = np.asarray(npz['cue_times']) if 'cue_times' in npz.files else np.arange(z_reactivation.shape[1])
    trial_index = np.asarray(meta['trial_index']).astype(int) if 'trial_index' in meta else np.arange(z_reactivation.shape[-1], dtype=int)
    memory_outcomes = np.asarray(meta['memory_outcomes'], dtype=object) if 'memory_outcomes' in meta else np.full(trial_index.shape, np.nan, dtype=object)
    if z_reactivation.ndim == 4:
        z_reactivation = np.nanmean(z_reactivation, axis=2)
    if projection_real is not None and projection_real.ndim == 4:
        projection_real = np.nanmean(projection_real, axis=2)
    return {
        'subject': subject,
        'path': mv_path,
        'convention': convention,
        'mv_predictor_assumed': bool(convention == 'legacy_unnamed'),
        'z_reactivation': z_reactivation,
        'projection_real': projection_real,
        'feedback_times': feedback_times,
        'cue_times': cue_times,
        'trial_index': trial_index,
        'memory_outcomes': memory_outcomes,
        'meta': meta,
    }

def interpolate_map_to_grid(map2d, src_fb, src_cue, tgt_fb, tgt_cue):
    if np.array_equal(src_fb, tgt_fb) and np.array_equal(src_cue, tgt_cue):
        return np.asarray(map2d, dtype=float)
    interp = RegularGridInterpolator((np.asarray(src_fb, dtype=float), np.asarray(src_cue, dtype=float)), np.asarray(map2d, dtype=float), bounds_error=False, fill_value=np.nan)
    fb_mesh, cue_mesh = np.meshgrid(np.asarray(tgt_fb, dtype=float), np.asarray(tgt_cue, dtype=float), indexing='ij')
    pts = np.column_stack([fb_mesh.ravel(), cue_mesh.ravel()])
    return interp(pts).reshape(len(tgt_fb), len(tgt_cue))


In [ ]:
# Smoke subject: load cue epochs and inspect ROI channels / indices
assert spectral_connectivity_time is not None, 'mne_connectivity.spectral_connectivity_time is required for this notebook'

smoke_subject = RUN['smoke_subject']
smoke_epoch_data = load_subject_epochs(
    subj_id=smoke_subject,
    base_dir=base_dir,
    event=GC_CFG['event'],
    roi_source=GC_CFG['roi_source'],
    roi_target=GC_CFG['roi_target'],
    tmin=GC_CFG['tmin'],
    tmax=GC_CFG['tmax'],
    target_sfreq=GC_CFG['target_sfreq'],
    day=GC_CFG['day'],
    fmin=GC_CFG['fmin'],
    fmax=GC_CFG['fmax'],
)
if smoke_epoch_data is None:
    raise RuntimeError(f'Failed to load epoch_data for {smoke_subject}')
smoke_epochs = smoke_epoch_data['epochs']
smoke_freqs = np.linspace(GC_CFG['fmin'], GC_CFG['fmax'], 8)

print('epochs shape:', smoke_epochs.get_data(copy=False).shape)
print('sfreq:', smoke_epochs.info['sfreq'])
print('times:', smoke_epochs.times[[0, -1]])
print('freqs:', smoke_freqs)
for hemi in ('l', 'r'):
    print('\nhemisphere:', hemi)
    print(' source channels:', smoke_epoch_data.get(f'source_channels_{hemi}', []))
    print(' target channels:', smoke_epoch_data.get(f'target_channels_{hemi}', []))
    print(' source indices:', smoke_epoch_data.get(f'source_indices_{hemi}', []))
    print(' target indices:', smoke_epoch_data.get(f'target_indices_{hemi}', []))


In [ ]:
# Smoke subject: build trial map, indices, and run forward/backward trialwise GC inline
trial_col = 'trials_mem' if 'trials_mem' in combined_df.columns else ('trials_gamble' if 'trials_gamble' in combined_df.columns else 'trials')
smoke_trial_map = combined_df[(combined_df['participant'] == smoke_subject) & (combined_df['condition'] == GC_CFG['day'])].copy()
smoke_trial_map = smoke_trial_map.dropna(subset=[trial_col]).copy()
smoke_trial_map['trial_index'] = pd.to_numeric(smoke_trial_map[trial_col], errors='coerce').astype(int) - 1
smoke_trial_map = smoke_trial_map[(smoke_trial_map['trial_index'] >= 0) & (smoke_trial_map['trial_index'] < len(smoke_epochs))].copy()
smoke_trial_map = smoke_trial_map.sort_values('trial_index').drop_duplicates(subset=['trial_index'], keep='first').reset_index(drop=True)
print('trial map rows:', len(smoke_trial_map))
display(smoke_trial_map[[c for c in ['participant', 'condition', trial_col, 'trial_index', 'rpe', 'rpe_cat', 'memory_outcome'] if c in smoke_trial_map.columns]].head())

smoke_indices_fwd_by_hemi = {}
smoke_indices_bwd_by_hemi = {}
smoke_gc_raw_by_hemi = {}

for hemi in ('l', 'r'):
    src_idx = np.unique(np.asarray(smoke_epoch_data.get(f'source_indices_{hemi}', []), dtype=int))
    tgt_idx = np.unique(np.asarray(smoke_epoch_data.get(f'target_indices_{hemi}', []), dtype=int))
    if src_idx.size == 0 or tgt_idx.size == 0:
        print(f'Skipping hemisphere {hemi}: missing source or target channels')
        continue

    indices_fwd = (np.array([src_idx.tolist()]), np.array([tgt_idx.tolist()]))
    indices_bwd = (np.array([tgt_idx.tolist()]), np.array([src_idx.tolist()]))
    smoke_indices_fwd_by_hemi[hemi] = indices_fwd
    smoke_indices_bwd_by_hemi[hemi] = indices_bwd

    # print(f'\n{hemi} indices_fwd:', indices_fwd)
    # print(f'{hemi} indices_bwd:', indices_bwd)

    gc_fwd = spectral_connectivity_time(
        smoke_epochs,
        sfreq=smoke_epochs.info['sfreq'],
        method=['gc'],
        indices=indices_fwd,
        fmin=GC_CFG['fmin'],
        fmax=GC_CFG['fmax'],
        freqs=smoke_freqs,
        n_cycles=GC_CFG['n_cycles'],
        gc_n_lags=GC_CFG['gc_n_lags'],
        faverage=True,
        verbose='ERROR',
    )
    gc_bwd = spectral_connectivity_time(
        smoke_epochs,
        sfreq=smoke_epochs.info['sfreq'],
        method=['gc'],
        indices=indices_bwd,
        fmin=GC_CFG['fmin'],
        fmax=GC_CFG['fmax'],
        freqs=smoke_freqs,
        n_cycles=GC_CFG['n_cycles'],
        gc_n_lags=GC_CFG['gc_n_lags'],
        faverage=True,
        verbose='ERROR',
    )

    gc_fwd_trial = extract_trial_gc_vector(gc_fwd, len(smoke_epochs), f'{hemi} forward')
    gc_bwd_trial = extract_trial_gc_vector(gc_bwd, len(smoke_epochs), f'{hemi} backward')
    gc_net_trial = gc_fwd_trial - gc_bwd_trial

    smoke_gc_raw_by_hemi[hemi] = {
        'gc_fwd_obj': gc_fwd,
        'gc_bwd_obj': gc_bwd,
        'gc_forward_theta_all': gc_fwd_trial,
        'gc_backward_theta_all': gc_bwd_trial,
        'gc_net_theta_all': gc_net_trial,
    }

    print(f'{hemi} first five forward:', gc_fwd_trial[:5])
    print(f'{hemi} first five backward:', gc_bwd_trial[:5])
    print(f'{hemi} first five net:', gc_net_trial[:5])


In [ ]:
# Smoke subject: source circular-shift surrogates inline, then combine hemispheres into gc_trial_df
smoke_gc_surrogates_by_hemi = {}
base_data = smoke_epochs.get_data(copy=True)
rng = np.random.default_rng(GC_CFG['random_state'])
min_shift = max(GC_CFG['gc_n_lags'] + 1, 2)

for hemi, hemi_raw in smoke_gc_raw_by_hemi.items():
    src_idx = np.unique(np.asarray(smoke_epoch_data[f'source_indices_{hemi}'], dtype=int))
    indices_fwd = smoke_indices_fwd_by_hemi[hemi]
    indices_bwd = smoke_indices_bwd_by_hemi[hemi]
    gc_forward_theta_all = hemi_raw['gc_forward_theta_all']
    gc_backward_theta_all = hemi_raw['gc_backward_theta_all']
    gc_net_theta_all = hemi_raw['gc_net_theta_all']

    n_trials = base_data.shape[0]
    n_times = base_data.shape[-1]
    candidate_shifts = np.arange(min_shift, n_times - min_shift) if (n_times - min_shift) > min_shift else np.arange(1, n_times)
    if candidate_shifts.size == 0:
        candidate_shifts = np.arange(1, n_times)

    surr_forward = np.full((GC_CFG['n_surrogates_smoke'], n_trials), np.nan, dtype=float)
    surr_backward = np.full((GC_CFG['n_surrogates_smoke'], n_trials), np.nan, dtype=float)
    surr_net = np.full((GC_CFG['n_surrogates_smoke'], n_trials), np.nan, dtype=float)

    print(f'\nRunning surrogates for hemisphere {hemi} ...')
    for s_idx in range(GC_CFG['n_surrogates_smoke']):
        shifted_data = base_data.copy()
        for trial_idx in range(n_trials):
            shift = int(rng.choice(candidate_shifts))
            shifted_data[trial_idx, src_idx, :] = np.roll(shifted_data[trial_idx, src_idx, :], shift=shift, axis=-1)
        shifted_epochs = mne.EpochsArray(
            shifted_data,
            info=deepcopy(smoke_epochs.info),
            events=smoke_epochs.events.copy(),
            event_id=smoke_epochs.event_id.copy() if smoke_epochs.event_id is not None else None,
            tmin=float(smoke_epochs.tmin),
            baseline=smoke_epochs.baseline,
            verbose=False,
        )
        surr_fwd = spectral_connectivity_time(
            shifted_epochs,
            sfreq=shifted_epochs.info['sfreq'],
            method=['gc'],
            indices=indices_fwd,
            fmin=GC_CFG['fmin'],
            fmax=GC_CFG['fmax'],
            freqs=smoke_freqs,
            n_cycles=GC_CFG['n_cycles'],
            gc_n_lags=GC_CFG['gc_n_lags'],
            faverage=True,
            verbose='ERROR',
        )
        surr_bwd = spectral_connectivity_time(
            shifted_epochs,
            sfreq=shifted_epochs.info['sfreq'],
            method=['gc'],
            indices=indices_bwd,
            fmin=GC_CFG['fmin'],
            fmax=GC_CFG['fmax'],
            freqs=smoke_freqs,
            n_cycles=GC_CFG['n_cycles'],
            gc_n_lags=GC_CFG['gc_n_lags'],
            faverage=True,
            verbose='ERROR',
        )
        surr_fwd_trial = extract_trial_gc_vector(surr_fwd, len(shifted_epochs), f'{hemi} surr forward {s_idx}')
        surr_bwd_trial = extract_trial_gc_vector(surr_bwd, len(shifted_epochs), f'{hemi} surr backward {s_idx}')
        surr_forward[s_idx] = surr_fwd_trial
        surr_backward[s_idx] = surr_bwd_trial
        surr_net[s_idx] = surr_fwd_trial - surr_bwd_trial

    smoke_gc_surrogates_by_hemi[hemi] = {
        'surr_forward': surr_forward,
        'surr_backward': surr_backward,
        'surr_net': surr_net,
        'gc_forward_theta_all': gc_forward_theta_all,
        'gc_backward_theta_all': gc_backward_theta_all,
        'gc_net_theta_all': gc_net_theta_all,
    }
    print('surrogate summary for first trial:', {'real_forward': gc_forward_theta_all[0], 'real_backward': gc_backward_theta_all[0], 'real_net': gc_net_theta_all[0], 'surr_net_mean': np.nanmean(surr_net[:, 0]), 'surr_net_std': np.nanstd(surr_net[:, 0])})

gc_rows = []
for _, row in smoke_trial_map.iterrows():
    trial_idx = int(row['trial_index'])
    hemi_rows = []
    for hemi in smoke_gc_surrogates_by_hemi:
        real_forward = float(smoke_gc_surrogates_by_hemi[hemi]['gc_forward_theta_all'][trial_idx])
        real_backward = float(smoke_gc_surrogates_by_hemi[hemi]['gc_backward_theta_all'][trial_idx])
        real_net = float(smoke_gc_surrogates_by_hemi[hemi]['gc_net_theta_all'][trial_idx])
        stats_fwd = scalar_null_stats(real_forward, smoke_gc_surrogates_by_hemi[hemi]['surr_forward'][:, trial_idx])
        stats_bwd = scalar_null_stats(real_backward, smoke_gc_surrogates_by_hemi[hemi]['surr_backward'][:, trial_idx])
        stats_net = scalar_null_stats(real_net, smoke_gc_surrogates_by_hemi[hemi]['surr_net'][:, trial_idx])
        hemi_rows.append({'hemisphere': hemi, 'gc_forward_theta': real_forward, 'gc_backward_theta': real_backward, 'gc_net_theta': real_net, 'gc_forward_z': stats_fwd['z'], 'gc_backward_z': stats_bwd['z'], 'gc_net_z': stats_net['z'], 'p_forward': stats_fwd['p'], 'p_backward': stats_bwd['p'], 'p_net': stats_net['p'], 'n_surrogates': stats_net['n']})
    hemi_df = pd.DataFrame(hemi_rows)
    gc_rows.append({'subject': smoke_subject, 'trial_index': trial_idx, 'trials_mem': int(row[trial_col]), 'rpe': row['rpe'] if 'rpe' in row else np.nan, 'rpe_cat': row['rpe_cat'] if 'rpe_cat' in row else np.nan, 'memory_outcome': row['memory_outcome'] if 'memory_outcome' in row else np.nan, 'gc_forward_theta': float(hemi_df['gc_forward_theta'].mean()), 'gc_backward_theta': float(hemi_df['gc_backward_theta'].mean()), 'gc_net_theta': float(hemi_df['gc_net_theta'].mean()), 'gc_forward_z': float(hemi_df['gc_forward_z'].mean()), 'gc_backward_z': float(hemi_df['gc_backward_z'].mean()), 'gc_net_z': float(hemi_df['gc_net_z'].mean()), 'p_forward': float(hemi_df['p_forward'].mean()), 'p_backward': float(hemi_df['p_backward'].mean()), 'p_net': float(hemi_df['p_net'].mean()), 'valid_hemi_count': int(len(hemi_df)), 'n_surrogates': int(np.round(hemi_df['n_surrogates'].mean())), 'gc_success': True})

smoke_gc_trial_df = pd.DataFrame(gc_rows).sort_values('trial_index').reset_index(drop=True)
smoke_gc_trial_df['gc_net_theta'] = smoke_gc_trial_df['gc_forward_theta'] - smoke_gc_trial_df['gc_backward_theta']
gc_trial_df = smoke_gc_trial_df.copy()
assert smoke_gc_trial_df['trial_index'].is_unique, 'gc_trial_df trial_index must be unique'
assert np.allclose(smoke_gc_trial_df['gc_net_theta'], smoke_gc_trial_df['gc_forward_theta'] - smoke_gc_trial_df['gc_backward_theta'], equal_nan=True)
assert smoke_gc_trial_df['p_net'].between(0, 1).all(), 'p_net must be in [0, 1]'
assert (smoke_gc_trial_df['n_surrogates'] > 0).all(), 'n_surrogates must be positive'
print('gc_trial_df rows:', len(gc_trial_df))
display(gc_trial_df.head())


In [ ]:
# Smoke subject: MV NPZ path fallback and explicit load diagnostics
smoke_mv_payload = load_subject_mv_payload(smoke_subject)
if smoke_mv_payload is None:
    raise FileNotFoundError('No MV NPZ found for smoke subject')
print('selected MV path:', smoke_mv_payload['path'])
print('MV convention:', smoke_mv_payload['convention'])
if smoke_mv_payload['mv_predictor_assumed']:
    print('WARNING: using legacy unnamed MV NPZ; predictor provenance is assumed')

smoke_z_reactivation = smoke_mv_payload['z_reactivation']
smoke_projection_real = smoke_mv_payload['projection_real']
smoke_feedback_times = smoke_mv_payload['feedback_times']
smoke_cue_times = smoke_mv_payload['cue_times']
smoke_trial_index = smoke_mv_payload['trial_index']
smoke_memory_outcomes = smoke_mv_payload['memory_outcomes']
smoke_meta = smoke_mv_payload['meta']

assert smoke_z_reactivation.ndim == 3, f'Expected 3D z_reactivation, got {smoke_z_reactivation.shape}'
assert smoke_z_reactivation.shape[-1] == len(smoke_trial_index), 'trial_index mismatch with z_reactivation'

mv_trial_df = pd.DataFrame({'subject': smoke_subject, 'trial_index': smoke_trial_index, 'trials_mem_mv': smoke_trial_index + 1, 'mv_trial_pos': np.arange(len(smoke_trial_index), dtype=int), 'memory_outcome_mv': smoke_memory_outcomes, 'mv_file_path': smoke_mv_payload['path'], 'mv_file_convention': smoke_mv_payload['convention'], 'mv_predictor_assumed': smoke_mv_payload['mv_predictor_assumed']})
print('z_reactivation shape:', smoke_z_reactivation.shape)
print('projection_real shape:', None if smoke_projection_real is None else smoke_projection_real.shape)
print('meta keys:', sorted(smoke_meta.keys()))
display(mv_trial_df.head())


In [ ]:
# Smoke subject: merge GC with MV, then inspect scalar-vs-scalar relationships
smoke_merged_trial_df = gc_trial_df.merge(mv_trial_df, on=['subject', 'trial_index'], how='inner')
merged_trial_df = smoke_merged_trial_df.copy()
assert len(smoke_merged_trial_df) > 0, 'Expected overlapping GC and MV trials'

trial_lookup = {int(t): i for i, t in enumerate(smoke_mv_payload['trial_index'])}
smoke_merged_trial_df['mv_trial_pos'] = smoke_merged_trial_df['trial_index'].map(trial_lookup).astype(int)
smoke_merged_trial_df['mv_mean_reinstatement'] = [float(np.nanmean(smoke_mv_payload['z_reactivation'][:, :, pos])) for pos in smoke_merged_trial_df['mv_trial_pos']]
smoke_merged_trial_df['mv_mid_cue_trace_mean'] = [float(np.nanmean(smoke_mv_payload['z_reactivation'][:, smoke_mv_payload['z_reactivation'].shape[1] // 2, pos])) for pos in smoke_merged_trial_df['mv_trial_pos']]
smoke_merged_trial_df['mv_mid_fb_trace_mean'] = [float(np.nanmean(smoke_mv_payload['z_reactivation'][smoke_mv_payload['z_reactivation'].shape[0] // 2, :, pos])) for pos in smoke_merged_trial_df['mv_trial_pos']]

print('merged rows:', len(smoke_merged_trial_df))
display(smoke_merged_trial_df.head())

fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
axes[0, 0].hist(gc_trial_df['gc_forward_theta'], bins=20, alpha=0.6, label='forward', color='#1f77b4')
axes[0, 0].hist(gc_trial_df['gc_backward_theta'], bins=20, alpha=0.6, label='backward', color='#d62728')
axes[0, 0].hist(gc_trial_df['gc_net_theta'], bins=20, alpha=0.5, label='net', color='#9467bd')
axes[0, 0].set_title('Trialwise theta GC')
axes[0, 0].legend()
axes[0, 1].hist(gc_trial_df['gc_net_z'], bins=20, color='#9467bd', alpha=0.8)
axes[0, 1].axvline(0, color='k', linestyle='--', linewidth=1)
axes[0, 1].set_title('Trialwise net GC z')
mean_map = np.nanmean(smoke_mv_payload['z_reactivation'], axis=2)
extent = [float(smoke_mv_payload['cue_times'].min()), float(smoke_mv_payload['cue_times'].max()), float(smoke_mv_payload['feedback_times'].min()), float(smoke_mv_payload['feedback_times'].max())]
vmax = np.nanmax(np.abs(mean_map))
if not np.isfinite(vmax) or vmax == 0:
    vmax = 1.0
im = axes[0, 2].imshow(mean_map, origin='lower', aspect='auto', extent=extent, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
axes[0, 2].set_title('Mean ACC z_reactivation')
axes[0, 2].set_xlabel('Cue time (s)')
axes[0, 2].set_ylabel('Feedback time (s)')
fig.colorbar(im, ax=axes[0, 2], shrink=0.8)
sns.regplot(data=smoke_merged_trial_df, x='gc_net_z', y='mv_mean_reinstatement', ax=axes[1, 0], scatter_kws={'s': 35})
axes[1, 0].set_title('Mean reinstatement vs gc_net_z')
sns.regplot(data=smoke_merged_trial_df, x='gc_net_z', y='mv_mid_cue_trace_mean', ax=axes[1, 1], scatter_kws={'s': 35})
axes[1, 1].set_title('Mid-cue slice mean vs gc_net_z')
sns.regplot(data=smoke_merged_trial_df, x='gc_net_z', y='mv_mid_fb_trace_mean', ax=axes[1, 2], scatter_kws={'s': 35})
axes[1, 2].set_title('Mid-feedback slice mean vs gc_net_z')
plt.show()


In [ ]:
# Smoke subject: subject-level GLM on full reinstatement map
smoke_glm_df = smoke_merged_trial_df[['trial_index', 'gc_net_z', 'mv_trial_pos']].copy()
smoke_glm_df = smoke_glm_df.dropna(subset=['gc_net_z']).sort_values('mv_trial_pos').drop_duplicates(subset=['trial_index'], keep='first').reset_index(drop=True)
smoke_glm_maps = np.transpose(smoke_mv_payload['z_reactivation'], (2, 0, 1))[smoke_glm_df['mv_trial_pos'].to_numpy(dtype=int)]
smoke_glm_predictors = smoke_glm_df[['gc_net_z']].copy()
assert smoke_glm_maps.shape[0] == len(smoke_glm_predictors), 'Map/predictor row mismatch'
assert len(smoke_glm_predictors) >= GLM_CFG['min_trials'], f"Need at least {GLM_CFG['min_trials']} aligned trials for the GLM"

glm = TFRClusterPermutationGLM(tfr_data=smoke_glm_maps, predictor_data=smoke_glm_predictors, target_var='gc_net_z', ch_name=f'{smoke_subject}:ACC_reinstatement', alternative=GLM_CFG['alternative'], alpha_cluster=GLM_CFG['alpha_cluster'], alpha_test=GLM_CFG['alpha_test'], standardize_X=True, random_state=GLM_CFG['seed'])
smoke_glm_result = glm.run(n_permutations=GLM_CFG['n_permutations_smoke'], n_jobs=1, verbose=0)
subject_glm_results[smoke_subject] = {'subject': smoke_subject, 'beta_map': smoke_glm_result['beta_map'], 't_map': smoke_glm_result['t_map'], 'clusters': smoke_glm_result['clusters'], 'cluster_pvals': np.asarray([c.pvalue for c in smoke_glm_result['clusters']], dtype=float), 'feedback_times': smoke_mv_payload['feedback_times'], 'cue_times': smoke_mv_payload['cue_times'], 'n_trials_used': int(len(smoke_glm_predictors)), 'skip_reason': ''}
print('n_trials_used:', len(smoke_glm_predictors))
print('n_clusters:', len(smoke_glm_result['clusters']))
print('n_significant_clusters:', len(smoke_glm_result['significant_clusters']))
print('cluster p-values:', [c.pvalue for c in smoke_glm_result['clusters']])


In [ ]:
# Smoke subject: GLM plots and interpretable summaries
beta_map = np.asarray(smoke_glm_result['beta_map'], dtype=float)
t_map = np.asarray(smoke_glm_result['t_map'], dtype=float)
fb_times = smoke_mv_payload['feedback_times']
cue_times = smoke_mv_payload['cue_times']
extent = [float(cue_times.min()), float(cue_times.max()), float(fb_times.min()), float(fb_times.max())]
sig_mask = np.zeros_like(beta_map, dtype=bool)
for cluster in smoke_glm_result['significant_clusters']:
    f0, f1 = cluster.freq_idx
    t0, t1 = cluster.time_idx
    sig_mask[f0:f1 + 1, t0:t1 + 1] = True
beta_vmax = np.nanmax(np.abs(beta_map))
t_vmax = np.nanmax(np.abs(t_map))
if not np.isfinite(beta_vmax) or beta_vmax == 0:
    beta_vmax = 1.0
if not np.isfinite(t_vmax) or t_vmax == 0:
    t_vmax = 1.0
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5), constrained_layout=True)
im0 = axes[0].imshow(beta_map, origin='lower', aspect='auto', extent=extent, cmap='RdBu_r', vmin=-beta_vmax, vmax=beta_vmax)
axes[0].set_title('Beta map')
axes[0].set_xlabel('Cue time (s)')
axes[0].set_ylabel('Feedback time (s)')
fig.colorbar(im0, ax=axes[0], shrink=0.8)
im1 = axes[1].imshow(t_map, origin='lower', aspect='auto', extent=extent, cmap='RdBu_r', vmin=-t_vmax, vmax=t_vmax)
axes[1].set_title('T map')
axes[1].set_xlabel('Cue time (s)')
axes[1].set_ylabel('Feedback time (s)')
fig.colorbar(im1, ax=axes[1], shrink=0.8)
im2 = axes[2].imshow(beta_map, origin='lower', aspect='auto', extent=extent, cmap='RdBu_r', vmin=-beta_vmax, vmax=beta_vmax)
if np.any(sig_mask):
    axes[2].contour(cue_times, fb_times, sig_mask.astype(int), levels=[0.5], colors='k', linewidths=1.25)
axes[2].set_title('Beta + significant clusters')
axes[2].set_xlabel('Cue time (s)')
axes[2].set_ylabel('Feedback time (s)')
fig.colorbar(im2, ax=axes[2], shrink=0.8)
plt.show()
beta_trace = np.nanmean(beta_map, axis=0)
t_trace = np.nanmean(t_map, axis=0)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5), constrained_layout=True)
axes[0].plot(cue_times, beta_trace, color='#2ca02c', linewidth=2)
axes[0].axhline(0, color='k', linestyle='--', linewidth=1)
axes[0].set_title('Feedback-averaged beta trace')
axes[0].set_xlabel('Cue time (s)')
axes[0].set_ylabel('Mean beta')
axes[1].plot(cue_times, t_trace, color='#ff7f0e', linewidth=2)
axes[1].axhline(0, color='k', linestyle='--', linewidth=1)
for start, end in contiguous_runs(np.nanmean(sig_mask.astype(int), axis=0) > 0):
    axes[1].axvspan(cue_times[start], cue_times[end], color='gray', alpha=0.2)
axes[1].set_title('Feedback-averaged t trace')
axes[1].set_xlabel('Cue time (s)')
axes[1].set_ylabel('Mean t')
plt.show()


In [ ]:
# Batch config and cache paths
RUN_BATCH_GC = True
RUN_BATCH_MERGE = True
RUN_BATCH_GLM = True
RUN_GROUP_GLM = True

OVERWRITE_BATCH_GC = True
OVERWRITE_BATCH_TABLES = True
OVERWRITE_BATCH_GLM = True
OVERWRITE_GROUP_GLM = True

combined_gc_csv = os.path.join(tables_dir, 'gc_trial_df_batch.csv')
combined_mv_csv = os.path.join(tables_dir, 'mv_trial_df_batch.csv')
combined_merged_csv = os.path.join(tables_dir, 'merged_trial_df_batch.csv')
subject_glm_summary_csv = os.path.join(tables_dir, 'subject_glm_summary.csv')
group_glm_npz = os.path.join(tables_dir, 'group_glm_summary.npz')
batch_gc_meta_json = os.path.join(tables_dir, 'batch_gc_meta.json')

print('batch_cache_dir:', batch_cache_dir)
print('RUN flags:', {'RUN_BATCH_GC': RUN_BATCH_GC, 'RUN_BATCH_MERGE': RUN_BATCH_MERGE, 'RUN_BATCH_GLM': RUN_BATCH_GLM, 'RUN_GROUP_GLM': RUN_GROUP_GLM})


In [ ]:
# Batch stage 1: all-subject trialwise GC with per-subject cache files
batch_gc_meta = {}
if RUN_BATCH_GC:
    for subject in tqdm(subjects, desc='Batch GC'):
        cache_csv = os.path.join(batch_cache_dir, f'{subject}_gc_trial_df.csv')
        cache_json = os.path.join(batch_cache_dir, f'{subject}_gc_meta.json')
        if os.path.exists(cache_csv) and os.path.exists(cache_json) and not OVERWRITE_BATCH_GC:
            with open(cache_json, 'r') as f:
                batch_gc_meta[subject] = json.load(f)
            continue
        try:
            subj_gc_df, subj_meta = compute_subject_gc_cache(subject, GC_CFG['n_surrogates_batch'], verbose=False)
        except Exception as exc:
            subj_gc_df = pd.DataFrame()
            subj_meta = {'subject': subject, 'skip_reason': str(exc)}
        if len(subj_gc_df) > 0:
            subj_gc_df.to_csv(cache_csv, index=False)
        with open(cache_json, 'w') as f:
            json.dump(subj_meta, f, indent=2, default=str)
        batch_gc_meta[subject] = subj_meta
else:
    for subject in subjects:
        cache_json = os.path.join(batch_cache_dir, f'{subject}_gc_meta.json')
        if os.path.exists(cache_json):
            with open(cache_json, 'r') as f:
                batch_gc_meta[subject] = json.load(f)

gc_frames = []
for subject in subjects:
    cache_csv = os.path.join(batch_cache_dir, f'{subject}_gc_trial_df.csv')
    if os.path.exists(cache_csv):
        gc_frames.append(pd.read_csv(cache_csv))
gc_trial_df_batch = pd.concat(gc_frames, ignore_index=True) if len(gc_frames) > 0 else pd.DataFrame()
if len(gc_trial_df_batch) > 0:
    assert not gc_trial_df_batch.duplicated(['subject', 'trial_index']).any(), 'Each per-subject GC cache must have unique trial_index'
    assert np.allclose(gc_trial_df_batch['gc_net_theta'], gc_trial_df_batch['gc_forward_theta'] - gc_trial_df_batch['gc_backward_theta'], equal_nan=True)
    assert gc_trial_df_batch['gc_success'].all(), 'Successful cache rows expected in batch GC table'
    assert gc_trial_df_batch['gc_net_z'].notna().all(), 'Successful rows must have finite gc_net_z'
    assert gc_trial_df_batch['p_net'].between(0, 1).all(), 'Batch p_net must be in [0, 1]'
    if RUN_BATCH_GC or OVERWRITE_BATCH_TABLES or (not os.path.exists(combined_gc_csv)):
        gc_trial_df_batch.to_csv(combined_gc_csv, index=False)
        with open(batch_gc_meta_json, 'w') as f:
            json.dump(batch_gc_meta, f, indent=2, default=str)
    gc_trial_df = gc_trial_df_batch.copy()

print('batch_gc subjects with cache:', len([s for s in subjects if os.path.exists(os.path.join(batch_cache_dir, f'{s}_gc_trial_df.csv'))]))
print('gc_trial_df batch rows:', len(gc_trial_df_batch))
if len(gc_trial_df_batch) > 0:
    display(gc_trial_df_batch.head())


In [ ]:
sns.barplot(data=gc_trial_df, x='rpe_cat', y='gc_net_z', errorbar='se')

In [ ]:
# Batch stage 2: all-subject MV discovery and merge
mv_rows = []
mv_payloads_batch = {}
mv_skip_rows = []
for subject in subjects:
    payload = load_subject_mv_payload(subject)
    if payload is None:
        mv_skip_rows.append({'subject': subject, 'skip_reason': 'missing_mv_npz'})
        continue
    if payload['mv_predictor_assumed']:
        print(f'WARNING: {subject} is using legacy unnamed MV NPZ: {payload["path"]}')
    assert payload['z_reactivation'].shape[-1] == len(payload['trial_index']), f'MV trial axis mismatch for {subject}'
    mv_payloads_batch[subject] = payload
    for pos, trial_idx in enumerate(payload['trial_index']):
        mv_rows.append({'subject': subject, 'trial_index': int(trial_idx), 'mv_trial_pos': int(pos), 'memory_outcome_mv': payload['memory_outcomes'][pos] if pos < len(payload['memory_outcomes']) else np.nan, 'mv_file_path': payload['path'], 'mv_file_convention': payload['convention'], 'mv_predictor_assumed': payload['mv_predictor_assumed']})

mv_trial_df_batch = pd.DataFrame(mv_rows)
if len(mv_trial_df_batch) > 0:
    mv_trial_df_batch = mv_trial_df_batch.sort_values(['subject', 'trial_index']).drop_duplicates(subset=['subject', 'trial_index'], keep='first').reset_index(drop=True)
    if RUN_BATCH_MERGE or OVERWRITE_BATCH_TABLES or (not os.path.exists(combined_mv_csv)):
        mv_trial_df_batch.to_csv(combined_mv_csv, index=False)
    mv_trial_df = mv_trial_df_batch.copy()

merged_trial_df_batch = pd.DataFrame()
if len(gc_trial_df_batch) > 0 and len(mv_trial_df_batch) > 0:
    merged_trial_df_batch = gc_trial_df_batch.merge(mv_trial_df_batch, on=['subject', 'trial_index'], how='inner')
    merged_trial_df_batch = merged_trial_df_batch.sort_values(['subject', 'trial_index']).drop_duplicates(subset=['subject', 'trial_index'], keep='first').reset_index(drop=True)
    if RUN_BATCH_MERGE or OVERWRITE_BATCH_TABLES or (not os.path.exists(combined_merged_csv)):
        merged_trial_df_batch.to_csv(combined_merged_csv, index=False)
    merged_trial_df = merged_trial_df_batch.copy()

batch_overlap_diag = pd.DataFrame({'gc_trials': gc_trial_df_batch.groupby('subject').size() if len(gc_trial_df_batch) > 0 else pd.Series(dtype=int), 'mv_trials': mv_trial_df_batch.groupby('subject').size() if len(mv_trial_df_batch) > 0 else pd.Series(dtype=int), 'merged_trials': merged_trial_df_batch.groupby('subject').size() if len(merged_trial_df_batch) > 0 else pd.Series(dtype=int)}).fillna(0).astype(int).reset_index().rename(columns={'index': 'subject'})
print('mv payloads loaded:', len(mv_payloads_batch))
print('mv_trial_df batch rows:', len(mv_trial_df_batch))
print('merged_trial_df batch rows:', len(merged_trial_df_batch))
if len(batch_overlap_diag) > 0:
    display(batch_overlap_diag.head(10))
if len(merged_trial_df_batch) > 0:
    display(merged_trial_df_batch.head())


In [ ]:
# Batch stage 3: subject-level GLMs on gc_net_z
subject_glm_summary_rows = []
if RUN_BATCH_GLM and len(merged_trial_df_batch) > 0:
    for subject in tqdm(sorted(merged_trial_df_batch['subject'].unique().tolist()), desc='Batch GLM'):
        subject_df = merged_trial_df_batch[merged_trial_df_batch['subject'] == subject].copy()
        payload = mv_payloads_batch.get(subject)
        result = {'subject': subject, 'skip_reason': ''}
        if payload is None:
            result['skip_reason'] = 'missing_mv_payload'
        else:
            subject_df = subject_df.dropna(subset=['gc_net_z']).sort_values('mv_trial_pos').drop_duplicates(subset=['trial_index'], keep='first').reset_index(drop=True)
            if len(subject_df) < GLM_CFG['min_trials']:
                result['skip_reason'] = f"aligned_trials<{GLM_CFG['min_trials']}"
            else:
                maps = np.transpose(payload['z_reactivation'], (2, 0, 1))[subject_df['mv_trial_pos'].to_numpy(dtype=int)]
                predictors = subject_df[[GLM_CFG['predictor_col']]].copy()
                try:
                    glm = TFRClusterPermutationGLM(tfr_data=maps, predictor_data=predictors, target_var=GLM_CFG['predictor_col'], ch_name=f'{subject}:ACC_reinstatement', alternative=GLM_CFG['alternative'], alpha_cluster=GLM_CFG['alpha_cluster'], alpha_test=GLM_CFG['alpha_test'], standardize_X=True, random_state=GLM_CFG['seed'])
                    glm_result = glm.run(n_permutations=GLM_CFG['n_permutations_batch'], n_jobs=1, verbose=0)
                    result.update({'beta_map': glm_result['beta_map'], 't_map': glm_result['t_map'], 'clusters': glm_result['clusters'], 'cluster_pvals': np.asarray([c.pvalue for c in glm_result['clusters']], dtype=float), 'feedback_times': payload['feedback_times'], 'cue_times': payload['cue_times'], 'n_trials_used': int(len(predictors)), 'skip_reason': ''})
                except Exception as exc:
                    result['skip_reason'] = str(exc)
        subject_glm_results[subject] = result
        sig_clusters = [] if result.get('skip_reason', '') != '' or 'clusters' not in result else [c for c in result['clusters'] if c.pvalue < GLM_CFG['alpha_test']]
        subject_glm_summary_rows.append({'subject': subject, 'status': 'ok' if result.get('skip_reason', '') == '' else 'skipped', 'skip_reason': result.get('skip_reason', ''), 'n_trials_used': int(result.get('n_trials_used', 0)), 'n_clusters': int(len(result.get('clusters', []))) if 'clusters' in result else 0, 'n_sig_clusters': int(len(sig_clusters)), 'min_sig_p': float(min([c.pvalue for c in sig_clusters])) if len(sig_clusters) > 0 else np.nan, 'max_sig_abs_mass': float(max([c.abs_mass for c in sig_clusters])) if len(sig_clusters) > 0 else np.nan})

subject_glm_summary = pd.DataFrame(subject_glm_summary_rows)
if len(subject_glm_summary) > 0 and (RUN_BATCH_GLM or OVERWRITE_BATCH_GLM or (not os.path.exists(subject_glm_summary_csv))):
    subject_glm_summary.to_csv(subject_glm_summary_csv, index=False)

print('subject_glm_results entries:', len(subject_glm_results))
if len(subject_glm_summary_rows) > 0:
    display(pd.DataFrame(subject_glm_summary_rows).head())


In [ ]:
# Batch stage 4: group-level one-sample cluster test on subject beta maps
if RUN_GROUP_GLM and len(subject_glm_results) > 0:
    valid_results = [(subject, res) for subject, res in subject_glm_results.items() if res.get('skip_reason', '') == '']
    included_subjects = []
    excluded_subjects = []
    beta_stack = []
    interpolated_subjects = []
    if len(valid_results) >= 2:
        ref_subject, ref_res = valid_results[0]
        ref_fb = np.asarray(ref_res['feedback_times'], dtype=float)
        ref_cue = np.asarray(ref_res['cue_times'], dtype=float)
        for subject, res in valid_results:
            beta_map = np.asarray(res['beta_map'], dtype=float)
            if not np.array_equal(np.asarray(res['feedback_times'], dtype=float), ref_fb) or not np.array_equal(np.asarray(res['cue_times'], dtype=float), ref_cue):
                beta_map = interpolate_map_to_grid(beta_map, res['feedback_times'], res['cue_times'], ref_fb, ref_cue)
                interpolated_subjects.append(subject)
            if np.isfinite(beta_map).all():
                beta_stack.append(beta_map)
                included_subjects.append(subject)
            else:
                excluded_subjects.append({'subject': subject, 'skip_reason': 'interpolated_beta_contains_nan'})
        if len(beta_stack) >= 2:
            beta_stack = np.asarray(beta_stack, dtype=float)
            t_obs, clusters, cluster_pvals, h0 = permutation_cluster_1samp_test(beta_stack, n_permutations=GLM_CFG['n_permutations_group'], tail=0, seed=GLM_CFG['seed'], verbose=False)
            sig_mask = np.zeros_like(t_obs, dtype=bool)
            for cluster, pval in zip(clusters, cluster_pvals):
                if pval < GLM_CFG['alpha_test']:
                    sig_mask[cluster] = True
            group_glm_summary = {'included_subjects': included_subjects, 'excluded_subjects': excluded_subjects, 'interpolated_subjects': interpolated_subjects, 't_obs': t_obs, 'clusters': clusters, 'cluster_pvals': cluster_pvals, 'sig_mask': sig_mask, 'feedback_times': ref_fb, 'cue_times': ref_cue}
            np.savez_compressed(group_glm_npz, t_obs=t_obs, cluster_pvals=np.asarray(cluster_pvals, dtype=float), sig_mask=sig_mask.astype(np.int8), feedback_times=ref_fb, cue_times=ref_cue, included_subjects=np.asarray(included_subjects, dtype=object), interpolated_subjects=np.asarray(interpolated_subjects, dtype=object))

if len(group_glm_summary) > 0:
    print('group included subjects:', group_glm_summary['included_subjects'])
    print('group interpolated subjects:', group_glm_summary['interpolated_subjects'])


In [ ]:
group_glm_summary

In [ ]:
# Batch QC cells
if len(gc_trial_df_batch) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(12, 9), constrained_layout=True)
    sns.violinplot(data=gc_trial_df_batch[['gc_forward_theta', 'gc_backward_theta', 'gc_net_theta']].melt(var_name='metric', value_name='value'), x='metric', y='value', ax=axes[0, 0], inner='box')
    axes[0, 0].set_title('All-subject GC distributions')
    axes[0, 0].tick_params(axis='x', rotation=25)
    axes[0, 1].hist(gc_trial_df_batch['gc_net_z'], bins=30, color='#9467bd', alpha=0.85)
    axes[0, 1].axvline(0, color='k', linestyle='--', linewidth=1)
    axes[0, 1].set_title('All-subject net GC z')
    if 'batch_overlap_diag' in globals() and len(batch_overlap_diag) > 0:
        plot_diag = batch_overlap_diag.sort_values('merged_trials', ascending=False)
        axes[1, 0].bar(np.arange(len(plot_diag)), plot_diag['merged_trials'].to_numpy(), color='#4c78a8')
        axes[1, 0].set_title('Merged trial counts by subject')
        axes[1, 0].set_xlabel('Subject rank')
        axes[1, 0].set_ylabel('Merged trials')
    else:
        axes[1, 0].text(0.5, 0.5, 'No merge diagnostics', ha='center', va='center', transform=axes[1, 0].transAxes)
    axes[1, 1].hist(gc_trial_df_batch['gc_forward_theta'], bins=25, alpha=0.6, label='forward', color='#1f77b4')
    axes[1, 1].hist(gc_trial_df_batch['gc_backward_theta'], bins=25, alpha=0.6, label='backward', color='#d62728')
    axes[1, 1].legend()
    axes[1, 1].set_title('Forward/backward descriptive QC')
    plt.show()

if len(group_glm_summary) > 0:
    mean_beta = np.nanmean(np.asarray([subject_glm_results[s]['beta_map'] for s in group_glm_summary['included_subjects']]), axis=0)
    extent = [float(group_glm_summary['cue_times'].min()), float(group_glm_summary['cue_times'].max()), float(group_glm_summary['feedback_times'].min()), float(group_glm_summary['feedback_times'].max())]
    vmax = np.nanmax(np.abs(mean_beta))
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
    im0 = axes[0].imshow(mean_beta, origin='lower', aspect='auto', extent=extent, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    axes[0].set_title('Group mean beta map')
    axes[0].set_xlabel('Cue time (s)')
    axes[0].set_ylabel('Feedback time (s)')
    fig.colorbar(im0, ax=axes[0], shrink=0.8)
    im1 = axes[1].imshow(group_glm_summary['t_obs'], origin='lower', aspect='auto', extent=extent, cmap='RdBu_r')
    if np.any(group_glm_summary['sig_mask']):
        axes[1].contour(group_glm_summary['cue_times'], group_glm_summary['feedback_times'], group_glm_summary['sig_mask'].astype(int), levels=[0.5], colors='k', linewidths=1.25)
    axes[1].set_title('Group t map + significant clusters')
    axes[1].set_xlabel('Cue time (s)')
    axes[1].set_ylabel('Feedback time (s)')
    fig.colorbar(im1, ax=axes[1], shrink=0.8)
    plt.show()

valid_batch_subjects = [s for s, res in subject_glm_results.items() if res.get('skip_reason', '') == '' and s != smoke_subject]
if len(valid_batch_subjects) > 0:
    example_subject = valid_batch_subjects[0]
    example_res = subject_glm_results[example_subject]
    extent = [float(example_res['cue_times'].min()), float(example_res['cue_times'].max()), float(example_res['feedback_times'].min()), float(example_res['feedback_times'].max())]
    vmax = np.nanmax(np.abs(example_res['beta_map']))
    if not np.isfinite(vmax) or vmax == 0:
        vmax = 1.0
    plt.figure(figsize=(5.5, 4.5))
    plt.imshow(example_res['beta_map'], origin='lower', aspect='auto', extent=extent, cmap='RdBu_r', vmin=-vmax, vmax=vmax)
    plt.title(f'Example batch subject beta map: {example_subject}')
    plt.xlabel('Cue time (s)')
    plt.ylabel('Feedback time (s)')
    plt.colorbar(shrink=0.8)
    plt.tight_layout()
    plt.show()
